# Restaurant Review Sentiment Analysis with a Local LLM

**Goal:** classify restaurant reviews as Positive, Negative or Neutral, extract aspect-level sentiment (food quality, ambience), and measure accuracy against star ratings.

- **Model:** `llama3.2:3b`, run locally via [Ollama](https://ollama.com), temperature 0
- **Data:** 20 restaurant reviews with star ratings (1, 3 or 5)
- **Approach:** prompt engineering only, no fine-tuning; every task is evaluated against ground truth

## 1. Setup and data

Load the reviews and check for missing values and the rating distribution.

In [1]:
import ollama, json

In [2]:
import pandas as pd
df = pd.read_csv("restaurant_reviews.csv")
print(df.shape)

(20, 3)


In [3]:
print(df.head())

  restaurant_ID  rating_review  \
0        FLV202              5   
1        SAV303              5   
2        YUM789              5   
3        TST101              5   
4        EAT456              5   

                                         review_full  
0  Totally in love with the Auro of the place, re...  
1  Kailash colony is brimming with small cafes no...  
2  Excellent taste and awesome decorum. Must visi...  
3  I have visited at jw lough/restourant. There w...  
4  Had a great experience in the restaurant food ...  


In [4]:
df.isnull().sum()

restaurant_ID    0
rating_review    0
review_full      0
dtype: int64

In [5]:
df.describe()

,rating_review
count,20.00000
mean,3.00000
std,1.71679
min,1.00000
25%,1.00000
50%,3.00000
75%,5.00000
max,5.00000


## 2. Task 1: free-text sentiment

`get_response` sends one prompt to the local model. The instruction and the review are separated by a blank line and a `Review:` label, so the model can tell where the instructions end and the review begins. Temperature 0 makes the output repeatable.

In [6]:
def get_response(instruction, content, label="Review"):
    prompt = f"{instruction}\n\n{label}: {content}"
    response = ollama.chat(model='llama3.2:3b', 
                           messages=[{'role':'user', 'content':prompt}],
                           options={"temperature": 0})
    return response['message']['content']

In [7]:
instruction1 = """Classify the sentiment of this restaurant review.
Choose between Positive, Negative or Neutral"""

df['result1'] = df['review_full'].apply(lambda x: get_response(instruction1, x))

In [8]:
df['result1'][0]

'I would classify the sentiment of this restaurant review as Positive. The reviewer uses words and phrases such as "Totally in love", "really beautiful", "quite fancy", "quaint and cute", "amazing", and "go to" to describe their experience at the restaurant. They also appreciate the restaurant\'s efforts to take precautions during the pandemic, which suggests that they value the restaurant\'s commitment to customer safety. Overall, the review is enthusiastic and recommends the restaurant to others.'

The answer is correct but buried in a paragraph, so a program has to search the text for the label. Task 2 asks for structured output instead.

## 3. Task 2: sentiment as JSON

Same task, but the prompt asks for JSON only. Structured output can be parsed with `json.loads` instead of searching prose for a label.

In [9]:
instruction2 = """Classify the sentiment of this restaurant review.
Return JSON only: {"sentiment": "Positive" | "Negative" | "Neutral"}"""

df['result2'] = df['review_full'].apply(lambda x: get_response(instruction2, x))

In [10]:
df['result2'][0]

'{"sentiment": "Positive"}'

In [11]:
df2 = pd.json_normalize(df['result2'].apply(lambda x: json.loads(x)))

In [12]:
df2.head()

,sentiment
0,Positive
1,Positive
2,Positive
3,Positive
4,Positive


In [13]:
df_result2 = pd.concat([df, df2], axis=1)
df_result2.head(1)

,restaurant_ID,rating_review,review_full,result1,result2,sentiment
0,FLV202,5,"Totally in love with the Auro of the place, re...",I would classify the sentiment of this restaur...,"{""sentiment"": ""Positive""}",Positive


JSON parsed cleanly for all 20 reviews.

## 4. Evaluation against star ratings

Star ratings serve as ground truth: 5 = Positive, 3 = Neutral, 1 = Negative.

In [14]:
label_map = {1: 'Negative', 3: 'Neutral', 5:'Positive'}
df['actual'] = df['rating_review'].map(label_map)

In [15]:
accuracy = (df['actual'] == df_result2['sentiment']).mean()
print(f'Accuracy: {accuracy:0.2%}')
print(pd.crosstab(df["actual"], df_result2['sentiment']))

Accuracy: 100.00%
sentiment  Negative  Neutral  Positive
actual                                
Negative          7        0         0
Neutral           0        6         0
Positive          0        0         7


**Result: 100% (20/20).** No misclassifications in any class.

## 5. Task 3: overall and aspect sentiment

One prompt returns three labels as JSON: overall sentiment, food quality and ambience.

In [16]:
instruction3 = """Classify the overall sentiment of this restaurant review along with the sentiment for food quality and ambience.
Classify only as "Positive", "Negative" or "Neutral"
Return the following keys in JSON format: {
        "Overall": "your_sentiment_prediction",
        "Food Quality": "your_sentiment_prediction",
        "Ambience": "your_sentiment_prediction"
    }
    Only return the json result and nothing else"""

df['result3'] = df['review_full'].apply(lambda x: get_response(instruction3, x))

In [17]:
df['result3'][0]

'{"Overall": "Positive", "Food Quality": "Positive", "Ambience": "Positive"}'

In [18]:
df3 = pd.json_normalize(df['result3'].apply(lambda x: json.loads(x)))

In [19]:
df_result3 = pd.concat([df, df3], axis=1)
df_result3.head(1)

,restaurant_ID,rating_review,review_full,result1,result2,actual,result3,Overall,Food Quality,Ambience
0,FLV202,5,"Totally in love with the Auro of the place, re...",I would classify the sentiment of this restaur...,"{""sentiment"": ""Positive""}",Positive,"{""Overall"": ""Positive"", ""Food Quality"": ""Posit...",Positive,Positive,Positive


### Evaluate overall sentiment against star ratings

Only the overall label has ground truth; the aspect labels are not scored.

In [20]:
accuracy = (df['actual'] == df_result3['Overall']).mean()
print(f'Accuracy: {accuracy:0.2%}')
print(pd.crosstab(df["actual"], df_result3['Overall']))

Accuracy: 100.00%
Overall   Negative  Neutral  Positive
actual                               
Negative         7        0         0
Neutral          0        6         0
Positive         0        0         7


In [21]:
df[df['actual'] != df_result3['Overall']]

,restaurant_ID,rating_review,review_full,result1,result2,actual,result3


In [22]:
df[df['actual'] != df_result3['Overall']]['review_full']

Series([], Name: review_full, dtype: str)

**Result: 100% (20/20).**

Without the blank line and `Review:` label between instruction and review, this prompt scored 95%: one 5-star review (TST101) was labelled Negative. Separating the instruction from the input fixed it.

## 6. Consistency check: LLM-as-judge vs code

Do the three tasks agree on overall sentiment? First, the model judges the three raw outputs.

In [23]:
instruction4 = """your goal is to check if the 3 results generated here are consistent with each other or not in terms of overall sentiment. Return result only as "Consistent" or "Not Consistent" and nothing else"""
df['consistency'] = df.apply(lambda x: get_response(instruction4, str({'result1': x['result1'], 'result2': x['result2'], 'result3':x['result3']}), label='Reviews'), axis=1)
df.head(2)

,restaurant_ID,rating_review,review_full,result1,result2,actual,result3,consistency
0,FLV202,5,"Totally in love with the Auro of the place, re...",I would classify the sentiment of this restaur...,"{""sentiment"": ""Positive""}",Positive,"{""Overall"": ""Positive"", ""Food Quality"": ""Posit...",Not Consistent
1,SAV303,5,Kailash colony is brimming with small cafes no...,I would classify the sentiment of this restaur...,"{""sentiment"": ""Positive""}",Positive,"{""Overall"": ""Positive"", ""Food Quality"": ""Posit...",Not Consistent


In [24]:
df['consistency'].value_counts()

consistency
Not Consistent    20
Name: count, dtype: int64

**Result: the judge called all 20 rows Not Consistent**, even though every set of labels agrees (shown below). The input is long, mixed prose and JSON, and a 3B model is a weak judge.

### Exact check in code

Extract the label from the Task 1 prose, then compare the three labels directly with pandas.

In [25]:
df['label1'] = df['result1'].apply(lambda t: next((s for s in ['Positive','Negative','Neutral'] if s in t), 'Unknown'))
df['consistent'] = (df['label1'] == df_result2['sentiment']) & (df_result2['sentiment'] == df_result3['Overall'])
print(df['consistent'].value_counts())

consistent
True    20
Name: count, dtype: int64


**Result: all 20 consistent.** This is an exact comparison, so code gives the right answer with no model call.

### Judge on clean labels

Does the judge do better if it only sees the three extracted labels?

In [26]:
df['consistency_clean'] = df.apply(lambda x: get_response(
    instruction4,
    f"{x['label1']}, {df_result2.loc[x.name, 'sentiment']}, {df_result3.loc[x.name, 'Overall']}",
    label="Labels"), axis=1)
print(df['consistency_clean'].value_counts())

consistency_clean
Consistent        13
Not Consistent     7
Name: count, dtype: int64


**Result: 13/20 Consistent.** Better than on raw outputs, but still wrong on 7 rows where the three labels are identical.

## Conclusion

- **Prompt formatting matters.** A blank line and a `Review:` label between instruction and review raised aspect-prompt accuracy from 95% to 100%.
- **Use code for exact checks.** An LLM judge checking whether three labels agree scored 0/20 on raw outputs and 13/20 on clean labels; a pandas comparison got 20/20. Save LLM-as-judge for open-ended outputs where no exact check exists.
- **Parse structured output, not prose.** JSON responses parsed cleanly for all 20 reviews.

**Limitations:** 20 short, clearly labelled reviews, so 100% here says little about harder data. Next step: a larger dataset and a comparison across models and prompts.